# 03 — HOMA-IR regression

Before HOMA-IR is thresholded into a binary insulin-resistance label, this
notebook asks how much of the index itself routine clinical variables can
explain. Six regressors are compared on two feature sets drawn from KNHANES, with
five-fold cross-validation and no hyperparameter tuning: the comparison is
between model families and between feature sets, not between tuned
configurations.

The target is the **natural logarithm** of HOMA-IR, so every R², MAE and RMSE
below is on the log scale. Exponentiating an error turns it into a
multiplicative factor on the HOMA-IR scale.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import polars as pl

from src.data.io import output_path, processed_path
from src.logging_utils import configure_logging
from src.models.regression import FEATURE_SETS, evaluate_feature_sets
from src.viz.tables import regression_latex_table

configure_logging(ROOT / "logs")

knhanes = pl.read_parquet(processed_path("KNHANES_data.parquet")).with_columns(RACE=2)
print(f"KNHANES {knhanes.shape}")
for name, columns in FEATURE_SETS.items():
    print(f"  {name}: {len(columns)} features -> {columns}")

KNHANES (15138, 22)
  KNHANES_9: 9 features -> ['AGE', 'BMI', 'FASTING_GLUCOSE', 'HBA1C', 'HDL_C', 'SEX', 'TG', 'T_CHO', 'RACE']
  KNHANES_17: 17 features -> ['AGE', 'BMI', 'FASTING_GLUCOSE', 'HBA1C', 'HDL_C', 'SEX', 'TG', 'T_CHO', 'BUN', 'CREATININE', 'URIC_ACID', 'LDL_C', 'SGOT', 'SGPT', 'MAP', 'BODY_WAISTLINE', 'RACE']


## Cross-validation

Six models × two feature sets × five folds. The splitter is built once and
reused, so every model sees the same folds. Takes one to two minutes; the
degree-2 polynomial expansion of 17 features is the slowest part.

In [2]:
results = evaluate_feature_sets(knhanes)
results

2026-09-22 19:45:58 [INFO] src.models.regression: KNHANES_9 | LinearRegression: R2=0.469 MAE=0.374 RMSE=0.490


2026-09-22 19:46:00 [INFO] src.models.regression: KNHANES_9 | Polynomial_Regression: R2=0.532 MAE=0.352 RMSE=0.460


2026-09-22 19:46:18 [INFO] src.models.regression: KNHANES_9 | RandomForest: R2=0.514 MAE=0.362 RMSE=0.469


2026-09-22 19:46:19 [INFO] src.models.regression: KNHANES_9 | XGBoost: R2=0.490 MAE=0.369 RMSE=0.480


2026-09-22 19:46:21 [INFO] src.models.regression: KNHANES_9 | LightGBM: R2=0.532 MAE=0.353 RMSE=0.460


...

2026-09-22 19:47:06 [INFO] src.models.regression: KNHANES_17 | RandomForest: R2=0.542 MAE=0.350 RMSE=0.455


2026-09-22 19:47:07 [INFO] src.models.regression: KNHANES_17 | XGBoost: R2=0.523 MAE=0.357 RMSE=0.465


2026-09-22 19:47:10 [INFO] src.models.regression: KNHANES_17 | LightGBM: R2=0.565 MAE=0.340 RMSE=0.443


2026-09-22 19:47:17 [INFO] src.models.regression: KNHANES_17 | CatBoost: R2=0.572 MAE=0.337 RMSE=0.440


model,feature_set,n_features,r2,mae,rmse
str,str,i64,f64,f64,f64
"""LinearRegression""","""KNHANES_9""",9,0.46944,0.373708,0.490024
"""Polynomial_Regression""","""KNHANES_9""",9,0.531747,0.352205,0.460283
"""RandomForest""","""KNHANES_9""",9,0.514236,0.361526,0.468783
"""XGBoost""","""KNHANES_9""",9,0.489977,0.368792,0.480382
"""LightGBM""","""KNHANES_9""",9,0.532288,0.35328,0.460016
…,…,…,…,…,…
"""Polynomial_Regression""","""KNHANES_17""",17,0.552519,0.340441,0.449719
"""RandomForest""","""KNHANES_17""",17,0.541544,0.34969,0.455475
"""XGBoost""","""KNHANES_17""",17,0.522535,0.35681,0.464807


## Outputs

Two files. `regression.xlsx` carries the numbers in tidy form, one row per model
and feature set. `regression_table.tex` carries the same numbers as LaTeX table
rows, ready to `\input`, so the typeset table and the spreadsheet cannot drift
apart.

In [3]:
results.to_pandas().to_excel(output_path("regression.xlsx"), index=False)

latex = regression_latex_table(results)
output_path("regression_table.tex").write_text(latex + "\n", encoding="utf-8")
print(latex)

2026-09-22 19:47:17 [INFO] src.viz.tables: Rendered LaTeX table with 6 rows


LinearRegression & \parbox[t]{3cm}{\linespread{1}\selectfont{$R^{2} = 0.469$}\\{$MAE=0.374$}\\{$RMSE=0.490$}} & \parbox[t]{3cm}{\linespread{1}\selectfont{$R^{2} = 0.499$}\\{$MAE=0.362$}\\{$RMSE=0.476$}} \\
\parbox[t]{5cm}{\linespread{1}\selectfont{Second-degree}\\{Polynomial Regression}} & \parbox[t]{3cm}{\linespread{1}\selectfont{$R^{2} = 0.532$}\\{$MAE=0.352$}\\{$RMSE=0.460$}} & \parbox[t]{3cm}{\linespread{1}\selectfont{$R^{2} = 0.553$}\\{$MAE=0.340$}\\{$RMSE=0.450$}} \\
RandomForest & \parbox[t]{3cm}{\linespread{1}\selectfont{$R^{2} = 0.514$}\\{$MAE=0.362$}\\{$RMSE=0.469$}} & \parbox[t]{3cm}{\linespread{1}\selectfont{$R^{2} = 0.542$}\\{$MAE=0.350$}\\{$RMSE=0.455$}} \\
XGBoost & \parbox[t]{3cm}{\linespread{1}\selectfont{$R^{2} = 0.490$}\\{$MAE=0.369$}\\{$RMSE=0.480$}} & \parbox[t]{3cm}{\linespread{1}\selectfont{$R^{2} = 0.523$}\\{$MAE=0.357$}\\{$RMSE=0.465$}} \\
LightGBM & \parbox[t]{3cm}{\linespread{1}\selectfont{$R^{2} = 0.532$}\\{$MAE=0.353$}\\{$RMSE=0.460$}} & \parbox[t]{3cm}{\li

### Reading the errors on the HOMA-IR scale

The best model is CatBoost on the 17-variable set. Because the target is
log-transformed, its errors exponentiate into multiplicative factors on the
HOMA-IR scale, which is the form worth quoting.

In [4]:
best = results.filter(
    (pl.col("model") == "CatBoost") & (pl.col("feature_set") == "KNHANES_17")
).to_dicts()[0]

print(f"R2   = {best['r2']:.3f}")
print(f"MAE  = {best['mae']:.3f} on the log scale -> a factor of {np.exp(best['mae']):.3f}")
print(f"RMSE = {best['rmse']:.3f} on the log scale -> a factor of {np.exp(best['rmse']):.3f}")

R2   = 0.572
MAE  = 0.337 on the log scale -> a factor of 1.400
RMSE = 0.440 on the log scale -> a factor of 1.552
